In [0]:
%sql
CREATE CATALOG IF NOT EXISTS vinkoscon
MANAGED LOCATION 'abfss://processeddatabricks@strfilese.dfs.core.windows.net/vinkos_managed/';

In [0]:
%sql
-- Forzamos el uso de tu nuevo catálogo
--USE CATALOG vinkoscon;

-- Creamos el esquema
--CREATE SCHEMA IF NOT EXISTS bronze;
--CREATE SCHEMA IF NOT EXISTS silver;
--CREATE SCHEMA IF NOT EXISTS gold;

-- Creamos la tabla (de raw)
--CREATE TABLE vinkoscon.bronze.visitas_raw
--PARTITIONED BY (create_date)
--AS SELECT *  FROM azure_sql_vinkos_catalog.dbo.usuarios_visitas_raw where create_date = date_format(current_timestamp());

INSERT INTO vinkoscon.bronze.visitas_raw (
email
,fechaPrimeraVisita
,fechaUltimaVisita
,visitasTotales
,visitasAnioActual
,visitasMesActual
,jyv
,Badmail
,Baja
,Fecha_envio
,Fecha_open
,Opens
,Opens_virales
,Fecha_click
,Clicks
,Clicks_virales
,Links
,IPs
,Navegadores
,Plataformas
,errores
,create_date
)
SELECT 
email
,fechaPrimeraVisita
,fechaUltimaVisita
,visitasTotales
,visitasAnioActual
,visitasMesActual
,jyv
,Badmail
,Baja
,Fecha_envio
,Fecha_open
,Opens
,Opens_virales
,Fecha_click
,Clicks
,Clicks_virales
,Links
,IPs
,Navegadores
,Plataformas
,errores
,cast(create_date as string) as create_date
FROM azure_sql_vinkos_catalog.dbo.usuarios_visitas_raw where cast(create_date as date) = current_date();

In [0]:
from pyspark.sql.functions import col, when, regexp_like, lit, trim ,to_timestamp ,to_date, date_format

#limpiamos el correo, aquel que no cumpla lo desechamos
df = spark.read.table("vinkoscon.bronze.visitas_raw")
emails_valido = df.filter(col("email").rlike(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')).select("*").where("CAST(create_date AS DATE) = CURRENT_DATE()")
sin_duplicados= emails_valido.dropDuplicates(['email'])
 
df_final = sin_duplicados.withColumn(
"fechaPrimeraVisita", 
    to_timestamp("fechaPrimeraVisita", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "fechaUltimaVisita", 
    col("fechaUltimaVisita").cast("string")
).withColumn(
    "visitasTotales",
    col("visitasTotales").cast("int")
).withColumn(
    "visitasAnioActual",
    col("visitasAnioActual").cast("int")
).withColumn(
    "visitasMesActual",
    col("visitasMesActual").cast("int")
).withColumn(
    "Clicks",
    col("Clicks").cast("int")
).withColumn(
    "Clicks_virales",
    col("Clicks_virales").cast("int")
).withColumn(
    "Opens",
    col("Opens").cast("int")
).withColumn(
    "Opens_virales",
    col("Opens_virales").cast("int")
).withColumn(
    "errores",
    col("errores").cast("int")
).withColumn(
    "Fecha_envio",
    to_timestamp("Fecha_envio", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "Fecha_open",
    to_timestamp("Fecha_open", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "Fecha_click",
    to_timestamp("Fecha_click", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "create_date",
    to_timestamp("create_date", "yyyy-MM-dd HH:mm:ss").cast("timestamp_ntz")
).withColumn(
    "jyv",
    when(col("jyv") == "S", True).when(col("jyv") == "N", False)
).withColumn(
    "Badmail",
    when(col("Badmail") == "S", True).when(col("Badmail") == "N", False)
).withColumn(
    "Baja",
    when(col("Baja") == "S", True).when(col("Baja") == "N", False)
)


df_visitante = df_final.select(col("email"),col("fechaPrimeraVisita"),col("fechaUltimaVisita"),col("visitasTotales"),col("visitasAnioActual"),col("visitasMesActual"))

df_estadistica=df_final.select(col("email"),col("jyv"),col("Badmail"),col("Baja"),col("Fecha_envio"),col("Fecha_open"),col("Opens"),col("Opens_virales"),col("Fecha_click"),col("Clicks"),col("Clicks_virales"),col("Links"),col("IPs"),col("Navegadores"),col("Plataformas"))

df_errores=df_final.select(col("email"),col("errores"),col("create_date"))


#df_visitante.write.mode("append").saveAsTable("vinkoscon.silver.visitas")
#df_estadistica.write.mode("append").saveAsTable("vinkoscon.silver.estadistica")
#df_errores.write.mode("append").saveAsTable("vinkoscon.silver.errores") 

